In [0]:
VOLUME_PATH = "/Volumes/workspace/default/m5_forecasting"

files = dbutils.fs.ls(VOLUME_PATH)
for f in files:
    print(f"{f.name:<40} {f.size / 1024 / 1024:>8.2f} MB")

In [0]:
calendar_df = spark.read.csv(
    f"{VOLUME_PATH}/calendar.csv",
    header=True,
    inferSchema=True
)

calendar_df.printSchema()

calendar_df.show(5)

print(f"Total rows: {calendar_df.count()}")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Define schema explicitly - we know what the columns are
prices_schema = StructType([
    StructField("store_id", StringType(), nullable=False),    # e.g. "CA_1", "TX_2"
    StructField("item_id", StringType(), nullable=False),     # e.g. "HOBBIES_1_001"
    StructField("wm_yr_wk", IntegerType(), nullable=False),   # Walmart week id
    StructField("sell_price", DoubleType(), nullable=True),   # Price in USD
])

# Read with explicit schema - much faster than inferSchema
prices_df = spark.read.csv(
    f"{VOLUME_PATH}/sell_prices.csv",
    header=True,
    schema=prices_schema
)

prices_df.printSchema()
prices_df.show(5)
print(f"Total rows: {prices_df.count()}")

In [0]:
# First, peek at the structure — read just the first row
sales_peek = spark.read.csv(
    f"{VOLUME_PATH}/sales_train_evaluation.csv",
    header=True,
    inferSchema=False  # Don't waste time inferring 1900+ columns
)

# How many columns?
print(f"Number of columns: {len(sales_peek.columns)}")

# What are the first 10 columns?
print(f"\nFirst 10 columns: {sales_peek.columns[:10]}")

# Last 5 columns?
print(f"Last 5 columns: {sales_peek.columns[-5:]}")

# Row count
print(f"\nRow count: {sales_peek.count()}")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Build schema: 6 metadata columns + 1941 day columns
metadata_fields = [
    StructField("id", StringType(), nullable=False),
    StructField("item_id", StringType(), nullable=False),
    StructField("dept_id", StringType(), nullable=False),
    StructField("cat_id", StringType(), nullable=False),
    StructField("store_id", StringType(), nullable=False),
    StructField("state_id", StringType(), nullable=False),
]

# Day columns: d_1, d_2, ..., d_1941 - all integers
day_fields = [
    StructField(f"d_{i}", IntegerType(), nullable=True) 
    for i in range(1, 1942)
]

sales_schema = StructType(metadata_fields + day_fields)

# Read with explicit schema - much faster than inferring 1947 columns
sales_wide = spark.read.csv(
    f"{VOLUME_PATH}/sales_train_evaluation.csv",
    header=True,
    schema=sales_schema
)

print(f"Rows: {sales_wide.count()}")
print(f"Columns: {len(sales_wide.columns)}")

# Show first few rows, but only first few columns (otherwise it overflows)
sales_wide.select("id", "item_id", "store_id", "state_id", "d_1", "d_2", "d_3").show(5)

In [0]:
# Identifier columns (stay as-is)
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

# Value columns (the 1941 day columns to unpivot)
day_cols = [f"d_{i}" for i in range(1, 1942)]

# Unpivot: wide -> long
sales_long = sales_wide.unpivot(
    ids=id_cols,
    values=day_cols,
    variableColumnName="d",        # Column name for the day labels (d_1, d_2, ...)
    valueColumnName="sales"        # Column name for the actual sales numbers
)

# Check the new shape
sales_long.printSchema()
sales_long.show(5)

In [0]:
from pyspark.sql.functions import broadcast

# Join sales with calendar on the "d" column
# Use broadcast() since calendar is tiny (~1969 rows)
sales_with_calendar = sales_long.join(
    broadcast(calendar_df),
    on="d",
    how="left"
)

# Check the result
sales_with_calendar.printSchema()

# Show a sample - pick columns that fit on screen
sales_with_calendar.select(
    "item_id", "store_id", "d", "date", "sales", 
    "weekday", "event_name_1", "snap_CA"
).show(5)

In [0]:
# Join with prices on (store_id, item_id, wm_yr_wk)
# Prices is too big to broadcast (~200MB), let Spark use sort-merge join
sales_full = sales_with_calendar.join(
    prices_df,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

# Schema check
print(f"Total columns: {len(sales_full.columns)}")
sales_full.printSchema()

# Sample rows - pick a product that's likely to have prices early on
sales_full.filter("sales > 0").select(
    "item_id", "store_id", "date", "sales", "sell_price", "wm_yr_wk"
).show(5)

In [0]:
# Write the joined dataset as a Delta table
TABLE_NAME = "workspace.default.m5_sales_full"

sales_full.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLE_NAME)

print(f"Saved to {TABLE_NAME}")

In [0]:
# Read from the Delta table - much faster than rebuilding the pipeline
sales = spark.table("workspace.default.m5_sales_full")

# Quick sanity check
print(f"Total rows: {sales.count():,}")
print(f"Columns: {len(sales.columns)}")

In [0]:
from pyspark.sql.functions import countDistinct, col

# Get the cardinality of categorical dimensions
sales.select(
    countDistinct("item_id").alias("unique_items"),
    countDistinct("store_id").alias("unique_stores"),
    countDistinct("state_id").alias("unique_states"),
    countDistinct("cat_id").alias("unique_categories"),
    countDistinct("dept_id").alias("unique_departments"),
    countDistinct("date").alias("unique_dates"),
).show()

In [0]:
from pyspark.sql.functions import min as spark_min, max as spark_max, datediff, lit

# Date range and span
sales.select(
    spark_min("date").alias("first_date"),
    spark_max("date").alias("last_date"),
    datediff(spark_max("date"), spark_min("date")).alias("span_days"),
).show()

In [0]:
from pyspark.sql import functions as F

# Sales distribution stats
sales.select(
    F.min("sales").alias("min"),
    F.max("sales").alias("max"),
    F.avg("sales").alias("mean"),
    F.stddev("sales").alias("stddev"),
    F.expr("percentile(sales, 0.5)").alias("median"),
    F.expr("percentile(sales, 0.95)").alias("p95"),
    F.expr("percentile(sales, 0.99)").alias("p99"),
).show()

# Zero-sales rate
total = sales.count()
zero_count = sales.filter(F.col("sales") == 0).count()

print(f"\nTotal rows: {total:,}")
print(f"Zero-sales rows: {zero_count:,} ({100 * zero_count / total:.1f}%)")

In [0]:
# Sales breakdown by category
sales.groupBy("cat_id").agg(
    F.count("*").alias("rows"),
    F.sum("sales").alias("total_units_sold"),
    F.avg("sales").alias("avg_sales_per_row"),
    F.sum(F.when(F.col("sales") > 0, 1).otherwise(0)).alias("non_zero_rows"),
).orderBy(F.desc("total_units_sold")).show()

In [0]:
# Price availability check
total = sales.count()
null_price = sales.filter(F.col("sell_price").isNull()).count()
zero_sales_null_price = sales.filter(
    (F.col("sell_price").isNull()) & (F.col("sales") == 0)
).count()
nonzero_sales_null_price = sales.filter(
    (F.col("sell_price").isNull()) & (F.col("sales") > 0)
).count()

print(f"Total rows:                              {total:,}")
print(f"Rows with NULL price:                    {null_price:,} ({100*null_price/total:.1f}%)")
print(f"  - of which had ZERO sales:             {zero_sales_null_price:,}")
print(f"  - of which had NON-ZERO sales:         {nonzero_sales_null_price:,}")

In [0]:
# Sales aggregated by year
sales.groupBy("year").agg(
    F.sum("sales").alias("total_units_sold"),
    F.countDistinct("date").alias("days_in_year"),
    F.count("*").alias("rows"),
).orderBy("year").show()

In [0]:
# Sales aggregated by month (across all years)
sales.groupBy("month").agg(
    F.sum("sales").alias("total_units"),
    F.avg("sales").alias("avg_sales_per_row"),
).orderBy("month").show()

In [0]:
# Filter out rows with no price (= product not yet on shelf)
sales_clean = sales.filter(F.col("sell_price").isNotNull())

# Confirm new size
new_total = sales_clean.count()
print(f"Before filter: {59_181_090:,}")
print(f"After filter:  {new_total:,}")
print(f"Dropped:       {59_181_090 - new_total:,} ({100*(59_181_090 - new_total)/59_181_090:.1f}%)")

In [0]:
from pyspark.sql.window import Window

# Define the window: partition by item-store, order by date
w = Window.partitionBy("item_id", "store_id").orderBy("date")

# Add lag features
sales_with_lags = sales_clean.withColumn(
    "sales_lag_7", F.lag("sales", 7).over(w)
).withColumn(
    "sales_lag_28", F.lag("sales", 28).over(w)
)

# Inspect - pick one item-store and see consecutive days
sales_with_lags.filter(
    (F.col("item_id") == "FOODS_3_090") & (F.col("store_id") == "CA_1")
).select("date", "sales", "sales_lag_7", "sales_lag_28") \
 .orderBy("date").show(35)

In [0]:
# Window for rolling: same partition+order as before, but with frame
# Frame is "previous 7 rows up to row before current" - excludes current row
w_roll_7 = Window.partitionBy("item_id", "store_id") \
                 .orderBy("date") \
                 .rowsBetween(-7, -1)

w_roll_28 = Window.partitionBy("item_id", "store_id") \
                  .orderBy("date") \
                  .rowsBetween(-28, -1)

# Add rolling features on top of lag features
sales_with_features = sales_with_lags.withColumn(
    "sales_rolling_mean_7", F.avg("sales").over(w_roll_7)
).withColumn(
    "sales_rolling_mean_28", F.avg("sales").over(w_roll_28)
).withColumn(
    "sales_rolling_std_7", F.stddev("sales").over(w_roll_7)
)

# Inspect same item-store, see rolling features alongside lags
sales_with_features.filter(
    (F.col("item_id") == "FOODS_3_090") & (F.col("store_id") == "CA_1")
).select(
    "date", "sales", "sales_lag_7", 
    "sales_rolling_mean_7", "sales_rolling_mean_28", "sales_rolling_std_7"
).orderBy("date").show(35)

In [0]:
sales_features = sales_with_features \
    .withColumn(
        # Pick the right SNAP column based on state
        "snap",
        F.when(F.col("state_id") == "CA", F.col("snap_CA"))
         .when(F.col("state_id") == "TX", F.col("snap_TX"))
         .when(F.col("state_id") == "WI", F.col("snap_WI"))
    ) \
    .withColumn(
        # Is there any event today?
        "has_event",
        F.when(F.col("event_name_1").isNotNull(), 1).otherwise(0)
    ) \
    .withColumn(
        # Weekend flag (Saturday=1, Sunday=2 in M5's wday)
        "is_weekend",
        F.when(F.col("wday").isin([1, 2]), 1).otherwise(0)
    ) \
    .withColumn(
        # Price last week (using the same window pattern)
        "price_lag_7", F.lag("sell_price", 7).over(w)
    ) \
    .withColumn(
        # Price change ratio: today vs last week
        "price_change_ratio",
        F.when(
            F.col("price_lag_7").isNotNull() & (F.col("price_lag_7") > 0),
            (F.col("sell_price") - F.col("price_lag_7")) / F.col("price_lag_7")
        ).otherwise(0)
    )

# Quick inspection
sales_features.filter(
    (F.col("item_id") == "FOODS_3_090") & (F.col("store_id") == "CA_1")
).select(
    "date", "sales", "snap", "has_event", "is_weekend", 
    "sell_price", "price_lag_7", "price_change_ratio"
).orderBy("date").show(10)

In [0]:
# Select final feature set
feature_cols = [
    # Identifiers (we keep these for grouping but won't feed to model directly)
    "date", "item_id", "store_id", "id",
    
    # Categorical features  
    "cat_id", "dept_id", "state_id",
    
    # Time features
    "wday", "month", "year", "is_weekend",
    
    # Lag features
    "sales_lag_7", "sales_lag_28",
    
    # Rolling features
    "sales_rolling_mean_7", "sales_rolling_mean_28", "sales_rolling_std_7",
    
    # Price features
    "sell_price", "price_lag_7", "price_change_ratio",
    
    # Event/SNAP features
    "snap", "has_event",
    
    # Target
    "sales",
]

# Build the final dataset
sales_final = sales_features.select(*feature_cols)

# Drop rows where lag_28 is null (insufficient history)
# These are the first 28 days for each item-store, ~28 * 30490 = ~854K rows
sales_final = sales_final.filter(F.col("sales_lag_28").isNotNull())

# Persist as Delta
FEATURES_TABLE = "workspace.default.m5_features"

sales_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(FEATURES_TABLE)

print(f"Saved to {FEATURES_TABLE}")